# ➰ Plotting Probabilities of Corrupted Model Runs ➰

## Parameters

In [1]:
model_id = "ai-forever/mGPT"
cache_dir = "/scratch/msonkin/word-order-thesis/cache/"

oneshot_template = ""
last_prompt_template = ""
num_shots = 1

data_path = "data/noun-adj.csv"
src_lang_base = "eng"
tgt_lang_base = "ger"
src_lang_source = "eng"
tgt_lang_source = "ger"

sample_size = 50
random_seed = 42

sentences_src_prefix_base = "phrase-"
sentences_tgt_prefix_base = "phrase-"
sentences_src_prefix_source = "phrase_head_final-"
sentences_tgt_prefix_source = "phrase_head_final-"

component_type = "block_output"

noun_base_prefix = "noun-"
adj_base_prefix = "adj-"
noun_source_prefix = "noun-"
adj_source_prefix = "adj_head_final-"

load_data = False # CRUCIAL: if True, doesn't run intervention, just loads existing data


block_intervention_save_path = None
head_intervention_save_path = None

hf_token = None

In [2]:
# Parameters
block_intervention_save_path = "output/intervention/probs/mlp_noun-adj_mgpt_eng-fre_eng-ger.csv"
model_id = "ai-forever/mGPT"
cache_dir = "/scratch/msonkin/word-order-thesis/cache/"
oneshot_template = "{lang_src}: \"{sentence_src}\" - {lang_tgt}: \"{sentence_tgt}\""
last_prompt_template = "{lang_src}: \"{sentence_src}\" - {lang_tgt}: \""
component_type = "mlp_output"
num_shots = 1
data_path = "data/noun-adj.csv"
src_lang_base = "eng"
tgt_lang_base = "fre"
src_lang_source = "eng"
tgt_lang_source = "ger"
sample_size = 199
random_seed = 42
sentences_src_prefix_base = "phrase-"
sentences_tgt_prefix_base = "phrase-"
sentences_src_prefix_source = "phrase-"
sentences_tgt_prefix_source = "phrase-"
noun_base_prefix = "noun-"
adj_base_prefix = "adj-"
noun_source_prefix = "noun-"
adj_source_prefix = "adj-"


In [3]:
if block_intervention_save_path is None:
    block_intervention_save_path = \
        f"output/intervention/block-intervention-results_{src_lang_base}-{tgt_lang_base}_{src_lang_source}-{tgt_lang_source}_{sentences_src_prefix_base}_{sentences_tgt_prefix_base}_{sentences_src_prefix_source}_{sentences_tgt_prefix_source}.csv"
if head_intervention_save_path is None:
    head_intervention_save_path = \
        f"output/intervention/head-intervention-results_{src_lang_base}-{tgt_lang_base}_{src_lang_source}-{tgt_lang_source}_{sentences_src_prefix_base}_{sentences_tgt_prefix_base}_{sentences_src_prefix_source}_{sentences_tgt_prefix_source}.csv"
block_intervention_plot_save_path = block_intervention_save_path.replace(".csv", ".png")
head_intervention_plot_save_path = head_intervention_save_path.replace(".csv", ".png")

shot_data_src_base = f'phrase-{src_lang_base}'
shot_data_tgt_base = f'phrase-{tgt_lang_base}'
shot_data_src_source = f'phrase-{src_lang_source}'
shot_data_tgt_source = f'phrase-{tgt_lang_source}'

## Setup

In [4]:
import torch
import pandas as pd
from tqdm import tqdm
import plotly.express as px
from huggingface_hub import login
from typing import List, Dict, Any
from statsmodels.formula.api import mixedlm
from transformers import AutoModelForCausalLM, AutoTokenizer

import pyvene as pv
from pyvene import embed_to_distrib, top_vals, format_token
from pyvene.models.modeling_utils import getattr_for_torch_module

from create_datasets.parallel_dataset import ParallelDataset
from utils.model_args import model_to_num_layers_attr, model_to_num_heads_attr

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
)
sm = torch.nn.Softmax(dim=2)

login(hf_token)

nnsight is not detected. Please install via 'pip install nnsight' for nnsight backend.


### Set up the Model

In [5]:
if not load_data:
    model = AutoModelForCausalLM.from_pretrained(model_id, cache_dir=cache_dir).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir=cache_dir)

    num_layers = getattr_for_torch_module(model, model_to_num_layers_attr[model_id])
    num_heads = getattr_for_torch_module(model, model_to_num_heads_attr[model_id])

### Set up the Data

In [6]:
if not load_data:
    df = pd.read_csv(data_path)

    base_df = df.sample(n=sample_size, random_state=random_seed).reset_index(drop=True)
    source_df = base_df.sample(n=sample_size, random_state=random_seed).reset_index(drop=True) # shuffled base

    base_dataset = ParallelDataset(
        model_id,
        dataframe=base_df,
        lang_src=src_lang_base,
        lang_tgt=tgt_lang_base,
        sentences_src_prefix=sentences_src_prefix_base,
        sentences_tgt_prefix=sentences_tgt_prefix_base,
        random_seed=random_seed,
    )

    source_dataset = ParallelDataset(
        model_id,
        dataframe=source_df,
        lang_src=src_lang_source,
        lang_tgt=tgt_lang_source,
        sentences_src_prefix=sentences_src_prefix_source,
        sentences_tgt_prefix=sentences_tgt_prefix_source,
        random_seed=random_seed,
    )

    base_prompts = base_dataset.format(
        oneshot_template,
        shots=num_shots,
        last_prompt_template=last_prompt_template,
        shot_data_src=shot_data_src_base,
        shot_data_tgt=shot_data_tgt_base,
        )
    print("=====Example of Base Prompt=====")
    print(base_prompts[0])

    base_tokens = base_dataset.prompts_to_tokens()

    source_prompts = source_dataset.format(
        oneshot_template,
        shots=num_shots,
        last_prompt_template=last_prompt_template,
        shot_data_src=shot_data_src_source,
        shot_data_tgt=shot_data_tgt_source,
        )
    print("=====Example of Source Prompt=====")
    print(source_prompts[0])

    source_tokens = source_dataset.prompts_to_tokens()

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/606 [00:00<?, ?B/s]

=====Example of Base Prompt=====
English: "national flag" - Français: "drapeau national"
English: "white goat" - Français: "
=====Example of Source Prompt=====
English: "round wheel" - Deutsch: "rundes Rad"
English: "cold mouth" - Deutsch: "


### Functions

#### Intervention Config

In [7]:
def intervention_config(model_type, intervention_type, unit, layer):
    """
    Parameters
    __________

    model_type: model type
    intervention_type: component for RepresentationConfig, e.g. head_attention_value_output
    unit: string to define the component type, e.g. "h" (head), "pos" (position), "h.pos" (head within position)
    later: layer id
    """
    # Set up the config to intervene
    config = pv.IntervenableConfig(
        model_type=model_type,
        representations=[
            pv.RepresentationConfig(
                layer,  # layer
                intervention_type,  # intervention type
                unit,  # intervention unit is now [pos] within [h]
                1,  # max number of unit
            ),
        ],
        intervention_types=pv.VanillaIntervention,
    )
    return config

#### Intervention Function

In [8]:
def intervention_data(
        base: torch.Tensor,
        source: torch.Tensor,
        base_pos: int,
        source_pos: int,
        tokentype2token: Dict[str, str],
        component_type: str,
        sentence_index: int = None,
        head_i: int = None,
        data: List[Dict[str, Any]] = None
    ) -> List[Dict[str, Any]]:
    """
    Collect intervention data for a given model component (block output or head attention value output).

    Parameters
    ----------
    base : torch.Tensor
        The tokenized base prompt tensor.
    source : torch.Tensor
        The tokenized source prompt tensor.
    base_pos : int
        The position of the last token in the base prompt.
    source_pos : int
        The position of the last token in the source prompt.
    tokentype2token : Dict[str, str]
        A dictionary mapping token types (e.g., 'noun-base', 'adj-base') to their corresponding token strings.
    component_type : str
        The type of model component to intervene on. Must be either 'block_output' or 'head_attention_value_output'.
    head_i : int, optional
        The index of the head to intervene on (if applicable). Only used for head-level interventions.
    data : List[Dict[str, Any]], optional
        A list to append the collected data to. If None, a new list will be created.

    Returns
    -------
    List[Dict[str, Any]]
        A list of dictionaries containing the intervention data, with keys:
        - "token_type": The type of token (e.g., 'noun-base', 'adj-base').
        - "token": The token string.
        - "prob": The probability of the token after intervention.
        - "layer": The layer index.
        - "head_i": The head index (if applicable).
        - "pos": The position index.
        - "type": The component type (e.g., 'block_output').

    Raises
    ------
    NotImplementedError
        If the component_type is not 'block_output' or 'head_attention_value_output'.
    """
    if data is None:
        data = []
    for layer_i in range(num_layers):
        if component_type in ["block_output", "mlp_output", "attention_value_output"]:
            unit = "pos"
        elif component_type == "head_attention_value_output":
            unit = "h.pos"
        else:
            raise NotImplementedError(f"Unsupported component type: {component_type}")
        config = intervention_config(
            type(model), component_type, unit, layer_i
        )
        intervenable = pv.IntervenableModel(config, model)
        if head_i is not None:
            unit_locations = {
                "sources->base": (
                    [[[[head_i]], [[source_pos]]]],  # intervene w/ target_head's pos_i
                    [[[[head_i]], [[base_pos]]]]
                ),
            }
        else:
            unit_locations = {"sources->base": (source_pos, base_pos)}
        # print(unit_locations)
        # print(base)
        # print(source)
        _, counterfactual_outputs = intervenable(
            base,
            source,
            unit_locations,
        )
        with torch.inference_mode():
            distrib = sm(counterfactual_outputs.logits)
        # print(f"\nTOP VALUES AT LAYER #{layer_i} AT POSITION {base_pos}:")
        # top_vals(tokenizer, distrib[0][base_pos], 5)
        for token_type, token in tokentype2token.items():
            # print(token_type, token, tokenizer.encode(token, add_special_tokens=False)[0], tokenizer.convert_ids_to_tokens(tokenizer.encode(token, add_special_tokens=False)[0]))
            data.append(
                {
                    "sentence_id": sentence_index,
                    "token_type": token_type,
                    "token": token,
                    "prob": float(distrib[0][base_pos][tokenizer.encode(token, add_special_tokens=False)[0],]),
                    "layer": layer_i,
                    "head_id": head_i,
                    "pos": base_pos,
                    "type": component_type,
                }
            )
    return data

## Block Output Intervention

### Calculating

In [9]:
if load_data:
    df = pd.read_csv(block_intervention_save_path)

else:
    data = []
    source_df_list = list(source_df.iterrows())
    for row_i, row in base_dataset.df.iterrows():
        # tokenize prompts
        prompt_base = tokenizer(base_prompts[row_i], return_tensors="pt").to(device)

        prompt_source = tokenizer(source_prompts[row_i], return_tensors="pt").to(device)
        # last token index
        pos_base = prompt_base.input_ids.size(1) - 1
        pos_source = prompt_source.input_ids.size(1) - 1
        # token type to token dict
        tokentype2token = {
            f"noun-base-{tgt_lang_base}": row[f'{noun_base_prefix}{tgt_lang_base}'],
            f"adj-base-{tgt_lang_base}": row[f'{adj_base_prefix}{tgt_lang_base}'],
            f"noun-source-{tgt_lang_base}": source_df_list[row_i][1][f'{noun_source_prefix}{tgt_lang_base}'],
            f"adj-source-{tgt_lang_base}": source_df_list[row_i][1][f'{adj_source_prefix}{tgt_lang_base}'],

            f"noun-base-{tgt_lang_source}": row[f'{noun_base_prefix}{tgt_lang_source}'],
            f"adj-base-{tgt_lang_source}": row[f'{adj_base_prefix}{tgt_lang_source}'],
            f"noun-source-{tgt_lang_source}": source_df_list[row_i][1][f'{noun_source_prefix}{tgt_lang_source}'],
            f"adj-source-{tgt_lang_source}": source_df_list[row_i][1][f'{adj_source_prefix}{tgt_lang_source}'],

            f"noun-base-{src_lang_base}": row[f'{noun_base_prefix}{src_lang_base}'],
            f"adj-base-{src_lang_base}": row[f'{adj_base_prefix}{src_lang_base}'],
            f"noun-source-{src_lang_base}": source_df_list[row_i][1][f'{noun_source_prefix}{src_lang_base}'],
            f"adj-source-{src_lang_base}": source_df_list[row_i][1][f'{adj_source_prefix}{src_lang_base}'],

            f"adj-base-rus": source_df_list[row_i][1][f'{adj_source_prefix}rus'],
            f"adj-base-ita": source_df_list[row_i][1][f'{adj_source_prefix}ita'],
            f"adj-base-ned": source_df_list[row_i][1][f'{adj_source_prefix}ned'],

            f"noun-base-rus": row[f'{noun_base_prefix}rus'],
            f"noun-base-ita": row[f'{noun_base_prefix}ita'],
            f"noun-base-ned": row[f'{noun_base_prefix}ned'],

        }
        # print(tokentype2token)

        data = intervention_data(
            prompt_base, 
            prompt_source, 
            pos_base,
            pos_source,
            tokentype2token,
            sentence_index=row_i,
            component_type=component_type,
            data=data
        )
    df = pd.DataFrame(data)
    df.to_csv("output/intervention/probs/attention_test.csv")

### Plotting

In [10]:
# Ensure the 'prob' column is numeric
df['prob'] = pd.to_numeric(df['prob'], errors='coerce')

# Drop rows with NaN in 'prob'
df = df.dropna(subset=['prob'])

# Create a line plot for token probabilities over layers
fig = px.line(
    df.groupby(['token_type', 'layer'], as_index=False)['prob'].mean(),
    x="layer",
    y="prob",
    color="token_type",
    title=f"Probabilities after Block Intervention ({src_lang_base}-{tgt_lang_base} base & {src_lang_source}-{tgt_lang_source} source)",
    labels={"layer": "Layer", "prob": "Probability", "token_type": "Token"},
    # category_orders={"layer": [str(i) for i in range(model.config.n_layer)]},
)

# Show the plot
fig.show()
# fig.write_image(block_intervention_plot_save_path)

## Head Intervention

### Calculating

In [11]:
# if load_data:
#     df = pd.read_csv(head_intervention_save_path)
# else:
#     data = []
#     source_df_list = list(source_df.iterrows())

#     for row_i, row in base_dataset.df.iterrows():
#         # tokenize prompts
#         prompt_base = tokenizer(base_prompts[row_i], return_tensors="pt").to(device)
#         prompt_source = tokenizer(source_prompts[row_i], return_tensors="pt").to(device)
#         # last token index
#         pos_base = prompt_base.input_ids.size(1) - 1
#         pos_source = prompt_source.input_ids.size(1) - 1
#         # token type to token dict
#         tokentype2token = {
#             f"noun-base-{tgt_lang_base}": row[f'{noun_base_prefix}{tgt_lang_base}'],
#             f"adj-base-{tgt_lang_base}": row[f'{adj_base_prefix}{tgt_lang_base}'],
#             f"noun-source-{tgt_lang_base}": source_df_list[row_i][1][f'{noun_source_prefix}{tgt_lang_base}'],
#             f"adj-source-{tgt_lang_base}": source_df_list[row_i][1][f'{adj_source_prefix}{tgt_lang_base}'],

#             f"noun-base-{tgt_lang_source}": row[f'{noun_base_prefix}{tgt_lang_source}'],
#             f"adj-base-{tgt_lang_source}": row[f'{adj_base_prefix}{tgt_lang_source}'],
#             f"noun-source-{tgt_lang_source}": source_df_list[row_i][1][f'{noun_source_prefix}{tgt_lang_source}'],
#             f"adj-source-{tgt_lang_source}": source_df_list[row_i][1][f'{adj_source_prefix}{tgt_lang_source}'],

#             f"noun-base-{src_lang_base}": row[f'{noun_base_prefix}{src_lang_base}'],
#             f"adj-base-{src_lang_base}": row[f'{adj_base_prefix}{src_lang_base}'],
#             f"noun-source-{src_lang_base}": source_df_list[row_i][1][f'{noun_source_prefix}{src_lang_base}'],
#             f"adj-source-{src_lang_base}": source_df_list[row_i][1][f'{adj_source_prefix}{src_lang_base}'],
#         }

#         for head_i in range(num_heads):
#             data = intervention_data(
#                 prompt_base, 
#                 prompt_source, 
#                 pos_base,
#                 pos_source,
#                 tokentype2token, 
#                 "head_attention_value_output", 
#                 sentence_index=row_i,
#                 head_i=head_i,
#                 data=data,
#             )
#     df = pd.DataFrame(data)
#     df.to_csv(head_intervention_save_path)

### Plotting

In [12]:
# print(df.head())

# # Ensure the 'prob' column is numeric
# df['prob'] = pd.to_numeric(df['prob'], errors='coerce')

# # Drop rows with NaN in 'prob'
# df = df.dropna(subset=['prob'])

# # Iterate over the unique tokens and create a heatmap for each
# tokens = df["token_type"].unique()
# for token in tokens:
#     token_df = df[df["token_type"] == token].groupby(['layer', 'head_id'], as_index=False).mean(numeric_only=True).reset_index()
#     heatmap_data = token_df.pivot(index="layer", columns="head_id", values="prob")

#     fig = px.imshow(
#         heatmap_data,
#         labels={"x": "Head", "y": "Layer", "color": "Probability"},
#         title=f"Probability Heatmap for Token after Head Intervention: {token}",
#         color_continuous_scale="viridis",
#     )
#     fig.show()

In [13]:
# HEAD 15.2 PREFERS NOUN-SOURCE MORE THAN ADJ-SOURCE?
# (difference not too big but it's interesting it is more visible. It sees the adjective too though.)

# Next up: Try with other languages as well (English-Italian, German-French)
# Bonus: Look into h.pos vs. pos: wtf are they doing?

In [14]:
# # Load the dataframe
# # df = pd.read_csv("data/head-intervention-results.csv")

# # Add a column to indicate if the head is 11.2
# df['is_head_11_2'] = (df['layer'] == 11) & (df['head_id'] == 2)

# # Filter the dataframe for noun-base-ger
# df_noun_base_ger = df[df['token_type'] == 'noun-base-ger']

# # Fit the mixed effects model
# model = mixedlm("prob ~ is_head_11_2", df_noun_base_ger, groups=df_noun_base_ger["sentence_id"])
# result = model.fit()

# # Print the summary of the model
# print(result.summary())
# print(result.pvalues)

# # Check if head 11.2 intervention leads to a significantly higher probability
# if result.pvalues['is_head_11_2[T.True]'] < 0.05:
#     print("Head 11.2 intervention leads to a significantly higher probability of noun-base-ger.")
# else:
#     print("Head 11.2 intervention does not lead to a significantly higher probability of noun-base-ger.")